[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/riccati/riccati.ipynb)

# Docking by Riccati Feedback

This example computes optimal feedback for docking a planar vessel under strong drag using the finite-horizon Riccati recursion. Reversible hull-mounted thrusters push and turn the vessel toward a target pose before a fixed deadline, trading quicker alignment along the approach against thruster effort.

Pose error is modelled as a twist, each thruster line of action as the join `mounts & directions` carrying both force and lever arm, and virtual work as their pairing `twist & forque`. Costs on errors and pushes are quadratic forms evaluated as `value(error, error)` and `effort_cost(push, push)`. Evaluating a quadratic form on linear maps pulls it back directly, so `value(dynamics, actuation)` couples the present error and push through the next step without transposes, and `-control_cost.solve(value(dynamics, actuation))` yields the feedback map from twists to forques in the backward sweep of the finite-horizon discrete-time Riccati recursion.

In matrix linear–quadratic regulator notation on $\mathrm{SE}(2)$, pose error $x = (\Delta p_x, \Delta p_y, \Delta\theta)^\top$ and wrench $u = (F_x, F_y, \tau)^\top$ require assembling lever-arm cross products $r_{i,x} d_{i,y} - r_{i,y} d_{i,x}$ into an allocation matrix $G$ with $R = (G W_u^{-1} G^\top)^{-1}$ and site-offset blocks $\begin{bmatrix} 1 & 0 & -s_{k,y} \\ 0 & 1 & s_{k,x} \end{bmatrix}$ into a readout matrix $C$ with $Q = C^\top W_x C$. Stepping backward from the terminal penalty, the [finite-horizon discrete-time Riccati recursion](https://ee363.stanford.edu/archive/lectures/dlqr.pdf#page=23) then forms $H_t = R + B^\top P_{t+1} B$, $K_t = -H_t^{-1} B^\top P_{t+1} A$, and $P_t = Q + A^\top P_{t+1}(A + B K_t)$ through explicit matrix transposes.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA2D
from examples.util.animation import save_figure, show_animation
from examples.mechanics.riccati import render

np.set_printoptions(precision=4, suppress=True)

ga = PGA2D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                             # Scalar+
Point = ga.gatype.antivector()                          # Antivector-
Motor = ga.gatype.rotor()                               # Rotor+
Twist = ga.gatype.bivector()                            # Bivector+
Forque = ga.gatype.antibivector()                       # Antibivector-
StateCost = ga.gatype((Scalar, Twist, Twist))           # Scalar+ <- (Twist+, Twist+)
EffortCost = ga.gatype((Scalar, Forque, Forque))        # Scalar+ <- (Forque-, Forque-)
Dynamics = ga.gatype((Twist, Twist))                    # Twist+ <- Twist+
Actuation = ga.gatype((Twist, Forque))                  # Twist+ <- Forque-
Feedback = ga.gatype((Forque, Twist))                   # Forque- <- Twist+

## Lines of force and docking geometry

Two thrusters push forward from opposite sides of the stern while a third pushes sideways near the bow. Joining each mounting point to its thrust direction `mounts & directions` forms a line of action in the plane. The term ***forque*** here means a wrench line carrying both a linear force and its moment about any reference point, paired against a displacement or velocity twist by `twist & forque` to measure virtual work.

In [ ]:
hull = mv.point([[-0.45, -0.23], [0.28, -0.23], [0.55, 0], [0.28, 0.23], [-0.45, 0.23]])  # [5] Point-
mounts = mv.point([[-0.3, 0.2], [-0.3, -0.2], [0.35, 0]])                                 # [3] Point-
# Ideal points represent unit thrust directions in the plane.
directions = mv.direction([[1, 0], [1, 0], [0, 1]])                                       # [3] Point-
# Bow and stern hull points fix both position and heading when aligned with the target.
tracking_points = mv.point([[0.45, 0], [-0.35, 0]])                                       # [2] Point-

In [ ]:
thrusters = mounts & directions                                                            # [3] Forque-
save_figure(render.setup_figure(hull, mounts, directions, tracking_points), "riccati_setup");

Grey markers and orange arrows show the three reversible thruster mounts and their positive thrust directions on the blue hull, while the two purple markers near the bow and stern locate the points whose displacement from the target pose is penalized.

## Quadratic cost forms and drag mobility

Summing the cost-weighted dyads `thrusters * (Twist & thrusters) / thruster_weights` maps any trial twist to the combined thruster response, and inverting that authority map prices the minimum-effort thruster allocation for any requested net forque through `resistance(Forque) & Forque`. Joining each tracking point to the coordinate directions forms four readout lines whose weighted squares penalize bow and stern displacement. Under strong drag, inertia drops out so the inverse drag map turns a net push directly into a pose increment over each local step.

In [ ]:
labels = ("Lower effort penalty", "Higher effort penalty")
n_cases = len(labels)
effort_scales = np.array([0.05, 5.0])                                                      # [n_cases]
thruster_weights = np.array([1.0, 2.0, 3.0])                                               # [3]
# Drag resists surge and sway through the centre along with yaw turning.
drag_weights = np.array([2.0, 3.0, 1.0])                                                   # [3]
# Rows weight bow and stern sites; columns weight forward and sideways alignment.
tracking_weights = np.array([[4.0, 8.0], [1.0, 2.0]])                                      # [2, 2]
dt = 0.1

In [ ]:
def tracking_cost(tracking_points: Point, axes: Point, tracking_weights: np.ndarray, dt: float) -> StateCost:
    """Sums weighted squared displacement readouts (`[] Scalar+ <- (Twist+, Twist+)`) over hull sites and axes."""
    tracking_lines = tracking_points[:, None] & axes[None, :]                 # [..., ...] Forque-
    readouts = Twist & tracking_lines                                         # [..., ...] Scalar+ <- Twist+
    return (readouts * readouts * tracking_weights).sum(axis=(0, 1)) * dt


def drag_mobility(centre: Point, axes: Point, drag_weights: np.ndarray) -> Actuation:
    """Inverts the translational and rotational drag dyad sum (`[] Twist+ <- Forque-`) into a planar mobility."""
    drag_lines = concatenate([centre & axes, (axes[0] & axes[1])[None]])      # [3] Forque-
    drag = (drag_lines * (Twist & drag_lines) * drag_weights).sum(axis=0)     # [] Forque- <- Twist+
    return drag.inverse()


# Dividing each thruster dyad by its command weight gives cheaper thrusters greater authority.
authority = (thrusters * (Twist & thrusters) / thruster_weights).sum(axis=0)  # [] Forque- <- Twist+
resistance = authority.inverse()                                              # [] Twist+ <- Forque-
effort_cost = (resistance(Forque) & Forque) * effort_scales * dt              # [n_cases] Scalar+ <- (Forque-, Forque-)
axes = mv.direction([[1, 0], [0, 1]])                                         # [2] Point-
centre = mv.point([0, 0])                                                     # [] Point-
state_cost = tracking_cost(tracking_points, axes, tracking_weights, dt)       # [] Scalar+ <- (Twist+, Twist+)
mobility = drag_mobility(centre, axes, drag_weights)                          # [] Twist+ <- Forque-
dynamics = Twist                                                              # [] Twist+ <- Twist+
actuation = mobility * dt                                                     # [] Twist+ <- Forque-

## Backward Riccati recursion

Starting from a stiff terminal pose penalty at the deadline, each backward step pulls the remaining cost form through `dynamics` and `actuation`. Because evaluating a bilinear form on linear maps composes them into its arguments, `value(actuation, actuation)` prices how a push moves the next error and `value(dynamics, actuation)` gives the cross-coupling whose linear solve determines the optimal feedback gain.

In [ ]:
n_steps = 72
# Pricing deadline error steeply enforces arrival at the target pose.
landing_weight = 1e6

In [ ]:
def riccati(value: StateCost, dynamics: Dynamics, actuation: Actuation, state_cost: StateCost,
            effort_cost: EffortCost, n_steps: int) -> Iterator[tuple[StateCost, Feedback]]:
    """Yields the remaining cost form and optimal feedback map (`[...]`) across `n_steps` backward steps."""
    for _ in range(n_steps):
        # A push pays effort now and shifts the next error inside the future cost form.
        control_cost = effort_cost + value(actuation, actuation)              # [...] Scalar+ <- (Forque-, Forque-)
        # Solving against the mixed pullback cancels the cost gradient for every error at once.
        feedback = -control_cost.solve(value(dynamics, actuation))            # [...] Forque- <- Twist+
        # Pulling the future cost through the closed-loop step updates the remaining cost form.
        value = state_cost + value(dynamics, dynamics + actuation(feedback))  # [...] Scalar+ <- (Twist+, Twist+)
        yield value, feedback


costs, gains = zip(*riccati(
    state_cost * landing_weight * np.ones(n_cases), dynamics, actuation, state_cost, effort_cost, n_steps,
))
values = stack(costs)                                                         # [n_steps, n_cases] Scalar+ <- (Twist+, Twist+)
# Reversing the backward sweep places the feedback policies in forward time order.
feedbacks = stack(gains)[::-1]                                                # [n_steps, n_cases] Forque- <- Twist+

save_figure(render.costs_figure(values), "riccati_costs");

The left and right panels show the lower and higher effort penalties across the zero-heading translation slice, where dark purple, teal, and light green contours trace the same remaining cost level with 1, 8, and 72 steps left before the deadline. More remaining steps expand the affordable region, while the higher effort penalty contracts every contour because thrust commands cost more.

## Forward rollout and cost accounting

Stepping forward from a shared initial offset and heading error applies each step's feedback gain to the current twist. Exponentiating `(-error / 2).exp()` recovers the rigid-body motor placing the hull relative to the target, while evaluating `resistance(pushes)` distributes each net forque across the three thrusters in inverse proportion to their command weights.

In [ ]:
# Both penalty cases start from the same translational and rotational offset.
initial_error = (mv.xw * 0.4 + mv.yw * 0.5 + mv.xy * 0.35) * np.ones(n_cases)  # [n_cases] Twist+

In [ ]:
def rollout(initial_error: Twist, dynamics: Dynamics, actuation: Actuation,
            feedbacks: Feedback) -> Iterator[Twist]:
    """Yields the initial pose error (`[...] Twist+`) and its forward trajectory under `feedbacks` (`[n_steps, ...]`)."""
    error = initial_error                                                     # [...] Twist+
    yield error
    for feedback in feedbacks:
        push = feedback(error)                                                # [...] Forque-
        error = dynamics(error) + actuation(push)                             # [...] Twist+
        yield error


errors = stack(tuple(rollout(initial_error, dynamics, actuation, feedbacks))) # [n_steps + 1, n_cases] Twist+
pushes = feedbacks(errors[:-1])                                               # [n_steps, n_cases] Forque-
# Pairing resistance(pushes) with each thruster line allocates the requested net forque at minimum effort.
commands = (resistance(pushes)[..., None] & thrusters) / thruster_weights     # [n_steps, n_cases, 3] Scalar+
poses = (errors * -0.5).exp()                                                 # [n_steps + 1, n_cases] Motor+
save_figure(render.approaches_figure(hull, poses), "riccati_approaches");

Both approaches reach the dashed target pose from the marked starting centre, with the lower effort penalty on the left aligning early and the higher effort penalty on the right spreading motion across the full horizon; faint hull outlines sample equal arc-length intervals along each path.

In [ ]:
tracking_paid = state_cost(errors[:-1], errors[:-1]).sum(axis=0)              # [n_cases] Scalar+
effort_paid = effort_cost(pushes, pushes).sum(axis=0)                         # [n_cases] Scalar+
landing_paid = state_cost(errors[-1], errors[-1]) * landing_weight            # [n_cases] Scalar+
predicted_cost = values[-1](initial_error, initial_error)                     # [n_cases] Scalar+
render.cost_summary(tracking_paid, effort_paid, landing_paid, predicted_cost, labels)

## Thruster commands along the approach

Carrying the hull and thruster mounts by the sequence of pose motors shows how the signed thruster commands evolve in body-fixed directions until the vessel comes to rest on the target at the deadline.

In [ ]:
frame_duration_ms = 40
arrow_scale = 0.04

In [ ]:
# Append a zero-command frame so the landed pose settles with thrusters off.
display_commands = concatenate((commands, commands[-1:] * 0))                 # [n_steps + 1, n_cases, 3] Scalar+
show_animation(render.docking_frames(hull, mounts, directions, poses, display_commands, arrow_scale),
               "riccati", frame_duration_ms)

Orange arrows drawn to one shared force scale reverse with command sign at each hull mount; the lower effort penalty on the left fires briskly near the start and coasts in, whereas the higher effort penalty on the right sustains gentler thrust up to the deadline.

In [ ]:
# checks
assert (effort_cost[:, None](thrusters, thrusters) > 0).all()
assert (state_cost(initial_error, initial_error) > 0).all()
assert (values[-1](initial_error, initial_error) < values[0](initial_error, initial_error)).all()
assert ((tracking_paid + effort_paid + landing_paid - predicted_cost).abs() < 1e-6).all()
forque_residual = (commands * thrusters).sum(axis=-1) - pushes                # [n_steps, n_cases] Forque-
assert ((resistance(forque_residual) & forque_residual).abs() < 1e-9).all()
landing_offsets = (poses[-1, :, None] >> tracking_points) - tracking_points   # [n_cases, 2] Point-
assert (landing_offsets.antinorm() < 1e-3).all()
assert (tracking_paid[0] < tracking_paid[1]).all()